In [ ]:
# Artefact locations, resolved from the installed package (hhb/paths.py), so
# this notebook runs correctly from any working directory:
#   CAT_FIG_DIR  -> figures/cat   (tracked: paper figures)
#   CAT_DATA_DIR -> data/cat      (tracked: saved solution branches)
#   MEDIA_DIR    -> media/        (gitignored: Wigner animations)
from hhb.paths import CAT_DATA_DIR, CAT_FIG_DIR, MEDIA_DIR

for _d in (CAT_FIG_DIR, CAT_DATA_DIR, MEDIA_DIR):
    _d.mkdir(parents=True, exist_ok=True)


In [ ]:
"""
Harmonic-balance solver for the two-mode SNAIL moment equations (EXACT,
all-orders in phia, phib -- no cubic-order Taylor truncation):

    dmu_a/dt        = -i*wa*mu_a + nl_mu_a
    dmu_b/dt        = -i*wb*mu_b - i*epsd*cos(wd t) - kappab/2*mu_b + nl_mu_b
    dsigma_a2/dt    =                                                nl_sa2      (no linear decay: mode a undamped)
    dsigma_b2/dt    = -kappab*sigma_b2 + kappab +                    nl_sb2
    dtsigma_a2/dt   = -2i*wa*tsigma_a2 +                             nl_ta2
    dtsigma_b2/dt   = -(kappab+2i*wb)*tsigma_b2 +                    nl_tb2
    dc/dt           = -(kappab/2+i*wa+i*wb)*c +                      nl_c
    dd/dt           = -(kappab/2+i*wa-i*wb)*d +                      nl_d

with (X_bar, K, P_a, P_a*, P_b, P_b* all built purely from the moments):
    Xbar  = phia*(mu_a+mu_a*) + phib*(mu_b+mu_b*)
    K     = phia^2*(sigma_a2+Re(tsigma_a2)-1/2) + phib^2*(sigma_b2+Re(tsigma_b2)-1/2)
            + 2*phia*phib*(Re(c)+Re(d))
    P_a   = phia*(sigma_a2+tsigma_a2)  + phib*(c+d)
    P_a*  = phia*(sigma_a2+tsigma_a2*) + phib*(c*+d*)
    P_b   = phib*(sigma_b2+tsigma_b2)  + phia*(c+d*)
    P_b*  = phib*(sigma_b2+tsigma_b2*) + phia*(c*+d)

    nl_mu_a = 2i*EJ*sinep*phia*(exp(-K)*cos(Xbar)-1)
    nl_mu_b = 2i*EJ*sinep*phib*(exp(-K)*cos(Xbar)-1)
    nl_sa2  =  2i*EJ*sinep*phia*exp(-K)*(P_a-P_a*)*sin(Xbar)
    nl_ta2  = -2i*EJ*sinep*phia*exp(-K)*(2*P_a-phia)*sin(Xbar)
    nl_sb2  =  2i*EJ*sinep*phib*exp(-K)*(P_b-P_b*)*sin(Xbar)
    nl_tb2  = -2i*EJ*sinep*phib*exp(-K)*(2*P_b-phib)*sin(Xbar)
    nl_c    = -2i*EJ*sinep*exp(-K)*(phib*P_a+phia*P_b-phia*phib)*sin(Xbar)
    nl_d    =  2i*EJ*sinep*exp(-K)*(phib*P_a-phia*P_b*)*sin(Xbar)

    (NOTE: these covariance nonlinear terms are pure sin(Xbar) -- the cos(Xbar)
    pieces cancel exactly against the mean-field's own contribution once the
    product rule d(tsigma_a2)/dt = d<a^2>/dt - 2*mu_a*dmu_a/dt etc. is applied
    correctly. An earlier version of this file had cos(Xbar) terms left in by
    mistake -- verified numerically against direct Lindblad simulation and
    corrected.)

State packed as n=14 real DOFs:
    y = [Re(mu_a), Im(mu_a), Re(mu_b), Im(mu_b), sigma_a2, sigma_b2,
         Re(tsigma_a2), Im(tsigma_a2), Re(tsigma_b2), Im(tsigma_b2),
         Re(c), Im(c), Re(d), Im(d)]

IMPORTANT: choose HBSettingsSnail.nu to match resonance conditions. If
wb = 2*wa and wd = wb (the standard cat-qubit two-photon-dissipation setup),
mode a's natural oscillation completes half a cycle per drive period, so the
true steady state has period 2*(2*pi/wd), not 2*pi/wd -- you need nu=2 (the
fundamental becomes wd/2 = wa). Using nu=1 in that case will simply fail to
converge (a safe failure mode, not a silently wrong answer).

KNOWN LIMITATION: for large-amplitude "cat" solutions (mu_a ~ 0, <a^2> large)
under the exact 2:1 resonance, the harmonic-balance Jacobian can become
nearly singular (a "cat orientation" flat direction) and plain Newton from a
naive initial guess may fail to converge. This is a continuation/regularization
issue, not a bug in the equations -- verified that N_time matches the exact
closed forms to machine precision, and that the solver converges cleanly on
generic (non-degenerate) parameter sets.
"""

import numpy as np
import sympy as sp
from scipy.sparse import lil_matrix, block_diag as sp_block_diag
from scipy.optimize import root
from dataclasses import dataclass


@dataclass
class SnailParams:
    wa: float
    wb: float
    phia: float
    phib: float
    EJ: float
    sinep: float     # sin(epsilon_p)
    epsd: float
    wd: float
    kappab: float


@dataclass
class HBSettingsSnail:
    N_H: int = 10
    samples_per_harmonic: int = 48
    nu: int = 1
    n: int = 14   # see state ordering in module docstring

    @property
    def N(self):
        return self.samples_per_harmonic * self.N_H


# ----------------------------------------------------------------------
# Symbolic derivation of the (purely nonlinear) RHS + its Jacobian.
# N(y) here needs ONLY phia, phib, EJ, sinep -- wa, wb, epsd, wd, kappab
# live entirely in Lin / b_ext.
# ----------------------------------------------------------------------
def _build_symbolic_nonlinear():
    (ma_r, ma_i, mb_r, mb_i, sa2, sb2, ta_r, ta_i, tb_r, tb_i,
     c_r, c_i, d_r, d_i) = sp.symbols(
        'ma_r ma_i mb_r mb_i sa2 sb2 ta_r ta_i tb_r tb_i c_r c_i d_r d_i',
        real=True)
    phia, phib, EJ, sinep = sp.symbols('phia phib EJ sinep', real=True)
    I = sp.I

    mu_a, mu_ac = ma_r + I * ma_i, ma_r - I * ma_i
    mu_b, mu_bc = mb_r + I * mb_i, mb_r - I * mb_i
    ta2, ta2c = ta_r + I * ta_i, ta_r - I * ta_i
    tb2, tb2c = tb_r + I * tb_i, tb_r - I * tb_i
    c_, cc_ = c_r + I * c_i, c_r - I * c_i
    d_, dc_ = d_r + I * d_i, d_r - I * d_i

    Xbar = phia * (mu_a + mu_ac) + phib * (mu_b + mu_bc)
    K = (phia**2 * (sa2 + ta_r - sp.Rational(1, 2))
         + phib**2 * (sb2 + tb_r - sp.Rational(1, 2))
         + 2 * phia * phib * (c_r + d_r))
    expmK = sp.exp(-K)
    cosX, sinX = sp.cos(Xbar), sp.sin(Xbar)

    P_a = phia * (sa2 + ta2) + phib * (c_ + d_)
    P_as = phia * (sa2 + ta2c) + phib * (cc_ + dc_)
    P_b = phib * (sb2 + tb2) + phia * (c_ + dc_)
    P_bs = phib * (sb2 + tb2c) + phia * (cc_ + d_)

    dmu_a = 2 * I * EJ * sinep * phia * (expmK * cosX - 1)
    dmu_b = 2 * I * EJ * sinep * phib * (expmK * cosX - 1)
    dsa2 = 2 * I * EJ * sinep * phia * expmK * (P_a - P_as) * sinX
    dta2 = -2 * I * EJ * sinep * phia * expmK * (2 * P_a - phia) * sinX
    dsb2 = 2 * I * EJ * sinep * phib * expmK * (P_b - P_bs) * sinX
    dtb2 = -2 * I * EJ * sinep * phib * expmK * (2 * P_b - phib) * sinX
    dc_eq = -2 * I * EJ * sinep * expmK * (phib * P_a + phia * P_b - phia * phib) * sinX
    dd_eq = 2 * I * EJ * sinep * expmK * (phib * P_a - phia * P_bs) * sinX

    F = sp.Matrix([
        sp.re(dmu_a), sp.im(dmu_a), sp.re(dmu_b), sp.im(dmu_b),
        sp.re(dsa2), sp.re(dsb2),
        sp.re(dta2), sp.im(dta2), sp.re(dtb2), sp.im(dtb2),
        sp.re(dc_eq), sp.im(dc_eq), sp.re(dd_eq), sp.im(dd_eq),
    ])
    y = sp.Matrix([ma_r, ma_i, mb_r, mb_i, sa2, sb2,
                   ta_r, ta_i, tb_r, tb_i, c_r, c_i, d_r, d_i])
    J = F.jacobian(y)

    params = (phia, phib, EJ, sinep)
    args = tuple(y) + params
    F_funcs = [sp.lambdify(args, F[i], modules='numpy') for i in range(14)]
    J_funcs = [[sp.lambdify(args, J[i, j], modules='numpy') for j in range(14)]
               for i in range(14)]
    return F_funcs, J_funcs


_F_SYM, _J_SYM = _build_symbolic_nonlinear()


def _complex_block(c):
    """2x2 real matrix representing multiplication by complex number c."""
    return np.array([[c.real, -c.imag], [c.imag, c.real]])


class SnailHillMethod:
    def __init__(self, snail: SnailParams, settings: HBSettingsSnail):
        self.p = snail
        self.s = settings
        self.n = settings.n
        self.N_H = settings.N_H
        self.N = settings.N
        self.dim = self.n * (2 * self.N_H + 1)

        self.tau_j = np.linspace(0, 2 * np.pi * settings.nu / snail.wd, self.N, endpoint=False)

        self.Lin = self._build_Lin()
        self.L = self.build_L()
        self.Gamma = self.build_Gamma()
        self.Gamma_pinv = np.linalg.pinv(self.Gamma, rcond=1e-12)
        self.b_ext = self.build_b_ext()

    def _build_Lin(self):
        p = self.p
        Lin = np.zeros((self.n, self.n))
        Lin[0:2, 0:2] = _complex_block(-1j * p.wa)                       # mu_a
        Lin[2:4, 2:4] = _complex_block(-1j * p.wb - p.kappab / 2)        # mu_b
        Lin[4, 4] = 0.0                                                   # sigma_a2 (undamped!)
        Lin[5, 5] = -p.kappab                                             # sigma_b2
        Lin[6:8, 6:8] = _complex_block(-2j * p.wa)                       # tsigma_a2
        Lin[8:10, 8:10] = _complex_block(-p.kappab - 2j * p.wb)          # tsigma_b2
        Lin[10:12, 10:12] = _complex_block(-p.kappab / 2 - 1j * (p.wa + p.wb))  # c
        Lin[12:14, 12:14] = _complex_block(-p.kappab / 2 - 1j * (p.wa - p.wb))  # d
        return Lin

    def build_L(self):
        n, N_H, nu = self.n, self.N_H, self.s.nu
        omega = self.p.wd
        dim = self.dim
        L = np.zeros((dim, dim))

        def idx_c0():
            return 0

        def idx_sk(k):
            return 1 + 2 * (k - 1)

        def idx_ck(k):
            return 2 + 2 * (k - 1)

        i0 = idx_c0()
        L[i0 * n:(i0 + 1) * n, i0 * n:(i0 + 1) * n] = -self.Lin

        for k in range(1, N_H + 1):
            freq = k * omega / nu
            is_, ic_ = idx_sk(k), idx_ck(k)
            L[is_ * n:(is_ + 1) * n, is_ * n:(is_ + 1) * n] = -self.Lin
            L[is_ * n:(is_ + 1) * n, ic_ * n:(ic_ + 1) * n] = -freq * np.eye(n)
            L[ic_ * n:(ic_ + 1) * n, is_ * n:(is_ + 1) * n] = freq * np.eye(n)
            L[ic_ * n:(ic_ + 1) * n, ic_ * n:(ic_ + 1) * n] = -self.Lin

        return L

    def build_Gamma(self):
        t, N_H, n, nu = self.tau_j, self.N_H, self.n, self.s.nu
        omega = self.p.wd
        k = np.arange(1, N_H + 1)
        S = np.sin(np.outer(t, k * omega / nu))
        C = np.cos(np.outer(t, k * omega / nu))
        Phi = np.empty((t.size, 2 * N_H + 1))
        Phi[:, 0] = 1.0 / np.sqrt(2.0)
        Phi[:, 1::2] = S
        Phi[:, 2::2] = C
        return np.kron(Phi, np.eye(n))

    def x_tilde_to_X(self, xt):
        return xt.reshape(self.N, self.n).T

    def X_to_x_tilde(self, X):
        return X.T.reshape(-1)

    def N_time(self, X):
        args_state = list(X)
        Ncol = X.shape[1]
        out = np.empty((self.n, Ncol))
        for i in range(self.n):
            val = _F_SYM[i](*args_state, self.p.phia, self.p.phib, self.p.EJ, self.p.sinep)
            out[i, :] = np.broadcast_to(val, (Ncol,))
        return out

    def dN_dX_blocks(self, X):
        args_state = list(X)
        Ncol = X.shape[1]
        Jarr = np.zeros((self.n, self.n, Ncol))
        for i in range(self.n):
            for j in range(self.n):
                val = _J_SYM[i][j](*args_state, self.p.phia, self.p.phib, self.p.EJ, self.p.sinep)
                Jarr[i, j, :] = np.broadcast_to(val, (Ncol,))
        return [Jarr[:, :, k] for k in range(Ncol)]

    #def build_dNtilde_dx_tilde(self, X):
    #    blocks = self.dN_dX_blocks(X)
    #    Jbig = lil_matrix((self.n * self.N, self.n * self.N))
    #    for j, Jj in enumerate(blocks):
    #        rows = slice(j * self.n, (j + 1) * self.n)
    #        Jbig[rows, rows] = Jj
    #    return Jbig.tocsr()
    def build_dNtilde_dx_tilde(self, X):
        blocks = self.dN_dX_blocks(X)
        return sp_block_diag(blocks, format='csr')

    def b_nl(self, z):
        X = self.x_tilde_to_X(self.Gamma @ z)
        Ftime = self.N_time(X)
        return self.Gamma_pinv @ self.X_to_x_tilde(Ftime)

    def db_dz(self, z):
        X = self.x_tilde_to_X(self.Gamma @ z)
        Jbig = self.build_dNtilde_dx_tilde(X)
        return self.Gamma_pinv @ (Jbig @ self.Gamma)

    def build_b_ext(self):
        n, N_H = self.n, self.N_H
        b = np.zeros(self.dim)

        def idx_c0():
            return 0

        def idx_ck(k):
            return 2 + 2 * (k - 1)

        i0 = idx_c0()
        b[i0 * n + 5] += self.p.kappab * np.sqrt(2.0)

        if N_H >= self.s.nu:
            icn = idx_ck(self.s.nu)
            b[icn * n + 3] += -self.p.epsd
        return b

    def residual(self, z):
        return self.L @ z - self.b_nl(z) - self.b_ext

    def jacobian(self, z):
        return self.L - self.db_dz(z)

    def solve(self, z0, **kwargs):
        sol = root(self.residual, z0, jac=self.jacobian, method='hybr', **kwargs)
        if not sol.success:
            raise RuntimeError(f"HB solve did not converge: {sol.message}")
        return sol.x

In [ ]:
sol_test_satellite_left = np.array([-9.06664921e-01,  7.34372449e-27, -8.40724897e-01, -7.97581254e-04,
        2.61745077e+01,  7.07028496e+00,  2.33204292e-01,  7.31808850e-16,
        2.13950970e-01,  5.56878929e-03,  2.21073724e-01,  2.06870794e-03,
        2.01761078e-01,  5.40385160e-03, -4.19259231e-01,  4.21663393e+00,
       -4.53150672e-17,  3.28002430e-16, -1.59295134e+00,  2.48808907e-02,
       -1.54780607e+00, -2.74767124e+00,  2.97576366e-02,  1.20772497e-01,
       -7.02305890e-01, -1.23442662e+00, -4.05156141e+00, -8.58170936e+00,
       -4.21663393e+00, -4.19259231e-01, -1.03678106e-16, -1.00700695e-16,
        2.71995078e+00, -7.14526271e-01,  2.77539169e+00, -1.57037870e+00,
        2.31643622e-01,  1.32015294e-02,  1.37247849e+00, -7.23608356e-01,
       -6.25992781e+00,  2.59231551e+00,  1.04225726e-03,  1.88717431e-02,
       -4.61830736e-01, -1.52832213e+00,  1.62004470e-01,  2.36888329e-02,
       -1.53429795e+01, -9.07736252e+00,  2.22689148e-01,  1.32250324e+00,
        4.90368416e-02,  5.83303368e-01,  3.66990374e-01, -6.32695177e-02,
       -9.43587156e-03,  2.08451453e-03,  1.52788400e+00, -4.60381258e-01,
        3.81137953e-01, -1.21462637e+00,  8.69622457e+00, -1.51809750e+01,
       -1.42991263e+00,  1.20680162e-01, -9.00842445e-01,  1.36113276e-01,
       -5.35268744e-03, -4.21995219e-01,  1.70183725e-02,  1.25394591e-01,
        1.01560630e-01,  3.71829711e-01,  4.85935247e-02,  4.16495073e-03,
       -3.63733328e-02,  4.40845435e-02,  1.63365869e-02, -1.18333861e-01,
        4.77398079e+00,  6.66199228e+00,  6.75322093e-02, -1.37524568e-01,
       -4.17981970e-02,  5.10551176e-02, -2.47822241e-01,  1.52105840e-01,
       -1.13428182e-01,  7.13543183e-02,  8.40384866e-02,  1.83302878e-02,
        8.64500959e-02,  1.55258596e-02, -6.57303036e+00,  4.81603033e+00,
       -1.31741408e-01, -6.37266039e-02, -5.39387774e-03, -4.30006352e-02,
       -2.50709913e-02, -9.96436799e-02,  1.29868560e-01,  5.71771716e-02,
       -2.43539083e-01, -3.40323572e-01, -7.32555176e-01, -3.70819648e+00,
       -4.49765572e-01, -6.13996133e-01,  1.14200122e-01,  1.49828350e-01,
        1.07501588e-02, -2.15755110e-02,  4.98099477e-02, -5.00947287e-02,
       -1.54643936e-01, -5.79022190e-02,  3.24805722e-01, -2.27341046e-01,
        3.76545798e+00, -6.71860710e-01,  6.19306378e-01, -4.18196842e-01,
       -1.21511363e-01,  8.67267747e-02,  7.90363476e-04,  7.32988195e-03,
        3.35593241e-03,  1.55311481e-02, -1.08141367e-01, -6.44207809e-02,
        1.60706881e-01, -7.57956885e-03,  1.52974457e-01,  6.68372887e-01,
        2.39042587e-01,  9.55719837e-02, -1.08882116e-01,  8.09137399e-02,
       -1.46597639e-03,  3.95181738e-03, -6.21118575e-03,  8.38393857e-03,
        6.40775744e-02,  1.78015244e-01, -6.10457469e-02,  1.31413784e-01,
       -7.12646346e-01,  1.10184899e-01, -1.64677355e-01,  2.02286055e-01,
        1.23566305e-01, -4.38661854e-02,  9.65481391e-04,  3.55056200e-03,
        3.91848902e-03,  7.19734630e-03,  2.87835500e-02,  6.70133484e-02,
       -3.26673428e-02,  1.12345964e-02, -1.76018913e-01, -1.40867377e-01,
       -6.84354078e-02, -2.25731713e-02,  4.80554451e-02, -4.69067368e-02,
       -5.91760334e-04,  5.79288835e-03, -2.39787630e-03,  1.17531922e-02,
       -5.05750625e-02, -6.11073305e-02,  4.68301970e-02, -1.16513782e-02,
        1.54949974e-01, -1.63419320e-01,  7.46482629e-02, -4.73293928e-02,
       -7.11859115e-02, -1.97149715e-02])

In [ ]:
sol_test_cat_neg = np.array([-9.77582673e-02, -7.72694453e-26, -9.06484935e-02, -8.59966672e-05,
        1.44530439e+00,  1.43920536e+00,  5.82484305e-03,  3.72803416e-17,
        9.58185591e-03,  3.27994287e-05,  7.04527305e-03,  1.29787658e-05,
        4.69075447e-04,  2.46788311e-05, -1.52432755e+00, -1.57051926e-02,
        5.92025729e-17,  2.30300705e-17,  4.15860915e-03, -7.19488924e-05,
        3.31131298e-02, -1.30831394e-04,  3.98453164e-02,  1.27744472e-04,
        3.21367367e-02, -7.18012744e-06, -3.43498425e-03, -8.15088876e-04,
        1.57051926e-02, -1.52432755e+00, -1.97546060e-17,  1.63302962e-17,
        2.78134136e-04,  5.66898837e-05, -1.64713481e-05,  1.86358695e-02,
       -4.16738532e-04,  9.94300030e-03, -1.90748325e-04,  1.19910665e-02,
       -5.48342451e-04,  7.27284952e-03, -1.64970936e-04,  1.94935010e-02,
       -5.19683057e-03,  2.41869290e-01, -3.76339637e-05, -9.67157574e-05,
        5.18981246e-03,  9.51134527e-02,  2.03064565e-04, -3.80256627e-03,
        1.07914189e-04, -1.64548035e-03, -4.11704904e-05,  2.59148517e-05,
       -9.74675050e-03, -3.29941873e-04, -2.41874220e-01, -5.42629248e-03,
       -3.03791129e-03, -2.89089184e-03, -9.20755414e-02,  5.15217850e-03,
        1.04962262e-02,  6.03894538e-05,  6.06465283e-03,  3.41443505e-05,
       -3.58341156e-03, -3.57621677e-05,  2.25781244e-03,  1.73423022e-04,
        1.33998118e-02,  4.81542096e-04, -1.21725343e-03,  4.27841206e-03,
        3.72821780e-03, -1.65346902e-04,  1.28761438e-02, -2.54380564e-04,
       -3.62277879e-02, -2.57558271e-03,  9.08208231e-04,  8.77364289e-05,
       -5.78076740e-05,  6.77343731e-03, -3.12553283e-04,  2.00994212e-02,
       -4.89337362e-05,  1.06552949e-04,  1.59165004e-04,  3.76644655e-03,
        2.54320113e-04,  1.28662593e-02,  2.57077674e-03, -3.62013581e-02,
        1.17134033e-05, -2.65023529e-03,  1.43251029e-05, -8.33787953e-05,
        6.62940544e-05, -1.93392085e-04,  6.58963179e-05,  7.14289236e-05,
       -1.33179564e-04,  9.06822895e-04,  1.39074628e-03,  7.74440484e-02,
       -3.23400344e-04,  1.69663940e-03,  7.41955839e-05, -3.17876721e-04,
        2.08446988e-05,  5.73004115e-05,  9.67274887e-05,  1.32679872e-04,
        4.77908775e-04,  2.18187014e-06, -9.31320223e-04, -1.34566492e-04,
       -7.74448431e-02,  1.38870668e-03, -1.71635470e-03, -3.25136357e-04,
        3.64287767e-04,  2.39753355e-05,  7.94451070e-06, -4.02137027e-06,
        3.36733693e-05, -8.56733242e-06, -1.88699054e-04, -2.28856554e-03,
        3.03982458e-04,  3.58865249e-05,  1.13228289e-02, -2.68693790e-05,
        1.94718979e-03,  5.03388938e-05, -9.18323090e-04, -1.56862731e-05,
        8.04274054e-07,  3.97225535e-05,  3.43971111e-06,  8.41866864e-05,
        2.21783946e-05,  2.54489954e-06, -3.65330046e-05,  2.88208511e-04,
        2.58071239e-05,  1.12928562e-02, -5.11842568e-05,  1.92530031e-03,
        1.82340932e-05,  6.31629927e-04, -2.53365915e-08, -1.08949777e-05,
       -1.08027250e-07, -2.20993463e-05, -3.12531117e-06, -6.67000896e-06,
        4.43620705e-06, -4.74514418e-05,  1.48559676e-05, -2.40527648e-03,
        8.62123446e-06, -4.92266499e-04, -4.72396371e-06, -2.37204501e-04,
        1.81582962e-06, -1.52019549e-07,  7.36641460e-06, -3.17093360e-07,
       -4.95252163e-05, -8.41955494e-04,  6.53423635e-05,  3.93268764e-06,
        2.44547833e-03,  1.38001725e-05,  5.19056025e-04,  7.89101162e-06,
       -3.12455627e-04, -1.02586118e-07])

In [ ]:
sol_test_satellite_right = np.array([-9.06664921e-01, -9.99865858e-25, -8.40724897e-01, -7.97581254e-04,
        2.61745077e+01,  7.07028496e+00,  2.33204292e-01,  4.90726957e-16,
        2.13950970e-01,  5.56878929e-03,  2.21073724e-01,  2.06870794e-03,
        2.01761078e-01,  5.40385160e-03,  4.19259231e-01, -4.21663393e+00,
        1.05842240e-15,  1.03153442e-15,  1.59295134e+00, -2.48808907e-02,
        1.54780607e+00,  2.74767124e+00, -2.97576366e-02, -1.20772497e-01,
        7.02305890e-01,  1.23442662e+00,  4.05156141e+00,  8.58170936e+00,
        4.21663393e+00,  4.19259231e-01, -1.19531604e-15,  3.37825278e-16,
       -2.71995078e+00,  7.14526271e-01, -2.77539169e+00,  1.57037870e+00,
       -2.31643622e-01, -1.32015294e-02, -1.37247849e+00,  7.23608356e-01,
        6.25992781e+00, -2.59231551e+00,  1.04225726e-03,  1.88717431e-02,
       -4.61830736e-01, -1.52832213e+00,  1.62004470e-01,  2.36888329e-02,
       -1.53429795e+01, -9.07736252e+00,  2.22689148e-01,  1.32250324e+00,
        4.90368416e-02,  5.83303368e-01,  3.66990374e-01, -6.32695177e-02,
       -9.43587156e-03,  2.08451453e-03,  1.52788400e+00, -4.60381258e-01,
        3.81137953e-01, -1.21462637e+00,  8.69622457e+00, -1.51809750e+01,
       -1.42991263e+00,  1.20680162e-01, -9.00842445e-01,  1.36113276e-01,
       -5.35268745e-03, -4.21995219e-01, -1.70183725e-02, -1.25394591e-01,
       -1.01560630e-01, -3.71829711e-01, -4.85935247e-02, -4.16495072e-03,
        3.63733328e-02, -4.40845435e-02, -1.63365869e-02,  1.18333861e-01,
       -4.77398079e+00, -6.66199228e+00, -6.75322093e-02,  1.37524568e-01,
        4.17981970e-02, -5.10551176e-02,  2.47822241e-01, -1.52105840e-01,
        1.13428182e-01, -7.13543183e-02, -8.40384866e-02, -1.83302878e-02,
       -8.64500959e-02, -1.55258596e-02,  6.57303036e+00, -4.81603033e+00,
        1.31741408e-01,  6.37266039e-02, -5.39387774e-03, -4.30006352e-02,
       -2.50709913e-02, -9.96436799e-02,  1.29868560e-01,  5.71771716e-02,
       -2.43539083e-01, -3.40323572e-01, -7.32555176e-01, -3.70819648e+00,
       -4.49765572e-01, -6.13996133e-01,  1.14200122e-01,  1.49828350e-01,
        1.07501588e-02, -2.15755110e-02,  4.98099477e-02, -5.00947287e-02,
       -1.54643936e-01, -5.79022190e-02,  3.24805722e-01, -2.27341046e-01,
        3.76545798e+00, -6.71860710e-01,  6.19306378e-01, -4.18196842e-01,
       -1.21511363e-01,  8.67267747e-02, -7.90363476e-04, -7.32988195e-03,
       -3.35593241e-03, -1.55311481e-02,  1.08141367e-01,  6.44207809e-02,
       -1.60706881e-01,  7.57956885e-03, -1.52974457e-01, -6.68372887e-01,
       -2.39042587e-01, -9.55719837e-02,  1.08882116e-01, -8.09137399e-02,
        1.46597639e-03, -3.95181738e-03,  6.21118575e-03, -8.38393857e-03,
       -6.40775744e-02, -1.78015244e-01,  6.10457469e-02, -1.31413784e-01,
        7.12646346e-01, -1.10184899e-01,  1.64677355e-01, -2.02286055e-01,
       -1.23566305e-01,  4.38661854e-02,  9.65481392e-04,  3.55056200e-03,
        3.91848902e-03,  7.19734630e-03,  2.87835500e-02,  6.70133484e-02,
       -3.26673428e-02,  1.12345964e-02, -1.76018913e-01, -1.40867377e-01,
       -6.84354078e-02, -2.25731713e-02,  4.80554451e-02, -4.69067368e-02,
       -5.91760334e-04,  5.79288835e-03, -2.39787630e-03,  1.17531922e-02,
       -5.05750625e-02, -6.11073305e-02,  4.68301970e-02, -1.16513782e-02,
        1.54949974e-01, -1.63419320e-01,  7.46482629e-02, -4.73293928e-02,
       -7.11859115e-02, -1.97149715e-02])

In [ ]:
sol_test_cat_pos = np.array([-9.77582673e-02,  2.30643165e-26, -9.06484935e-02, -8.59966672e-05,
        1.44530439e+00,  1.43920536e+00,  5.82484305e-03,  5.15742963e-17,
        9.58185591e-03,  3.27994287e-05,  7.04527305e-03,  1.29787658e-05,
        4.69075447e-04,  2.46788311e-05,  1.52432755e+00,  1.57051926e-02,
        2.59369764e-18,  3.30364933e-17, -4.15860915e-03,  7.19488924e-05,
       -3.31131298e-02,  1.30831394e-04, -3.98453164e-02, -1.27744472e-04,
       -3.21367367e-02,  7.18012744e-06,  3.43498425e-03,  8.15088876e-04,
       -1.57051926e-02,  1.52432755e+00, -6.61786409e-17,  8.06913120e-18,
       -2.78134136e-04, -5.66898838e-05,  1.64713481e-05, -1.86358695e-02,
        4.16738532e-04, -9.94300030e-03,  1.90748325e-04, -1.19910665e-02,
        5.48342451e-04, -7.27284952e-03, -1.64970936e-04,  1.94935010e-02,
       -5.19683057e-03,  2.41869290e-01, -3.76339637e-05, -9.67157574e-05,
        5.18981246e-03,  9.51134527e-02,  2.03064565e-04, -3.80256627e-03,
        1.07914189e-04, -1.64548035e-03, -4.11704904e-05,  2.59148517e-05,
       -9.74675050e-03, -3.29941873e-04, -2.41874220e-01, -5.42629248e-03,
       -3.03791129e-03, -2.89089185e-03, -9.20755414e-02,  5.15217850e-03,
        1.04962262e-02,  6.03894538e-05,  6.06465283e-03,  3.41443505e-05,
       -3.58341156e-03, -3.57621677e-05, -2.25781244e-03, -1.73423022e-04,
       -1.33998118e-02, -4.81542096e-04,  1.21725343e-03, -4.27841206e-03,
       -3.72821780e-03,  1.65346902e-04, -1.28761438e-02,  2.54380564e-04,
        3.62277879e-02,  2.57558271e-03, -9.08208231e-04, -8.77364289e-05,
        5.78076740e-05, -6.77343731e-03,  3.12553283e-04, -2.00994212e-02,
        4.89337362e-05, -1.06552949e-04, -1.59165004e-04, -3.76644655e-03,
       -2.54320113e-04, -1.28662593e-02, -2.57077674e-03,  3.62013581e-02,
       -1.17134033e-05,  2.65023529e-03,  1.43251029e-05, -8.33787953e-05,
        6.62940544e-05, -1.93392085e-04,  6.58963179e-05,  7.14289236e-05,
       -1.33179564e-04,  9.06822895e-04,  1.39074628e-03,  7.74440484e-02,
       -3.23400344e-04,  1.69663940e-03,  7.41955839e-05, -3.17876721e-04,
        2.08446988e-05,  5.73004115e-05,  9.67274887e-05,  1.32679872e-04,
        4.77908775e-04,  2.18187014e-06, -9.31320223e-04, -1.34566492e-04,
       -7.74448431e-02,  1.38870668e-03, -1.71635470e-03, -3.25136357e-04,
        3.64287767e-04,  2.39753355e-05, -7.94451070e-06,  4.02137027e-06,
       -3.36733693e-05,  8.56733242e-06,  1.88699054e-04,  2.28856554e-03,
       -3.03982458e-04, -3.58865249e-05, -1.13228289e-02,  2.68693789e-05,
       -1.94718979e-03, -5.03388938e-05,  9.18323090e-04,  1.56862730e-05,
       -8.04274054e-07, -3.97225535e-05, -3.43971111e-06, -8.41866864e-05,
       -2.21783946e-05, -2.54489955e-06,  3.65330046e-05, -2.88208511e-04,
       -2.58071239e-05, -1.12928562e-02,  5.11842568e-05, -1.92530031e-03,
       -1.82340932e-05, -6.31629927e-04, -2.53365915e-08, -1.08949777e-05,
       -1.08027250e-07, -2.20993463e-05, -3.12531117e-06, -6.67000896e-06,
        4.43620705e-06, -4.74514418e-05,  1.48559676e-05, -2.40527648e-03,
        8.62123446e-06, -4.92266499e-04, -4.72396371e-06, -2.37204501e-04,
        1.81582962e-06, -1.52019549e-07,  7.36641460e-06, -3.17093360e-07,
       -4.95252163e-05, -8.41955494e-04,  6.53423635e-05,  3.93268764e-06,
        2.44547833e-03,  1.38001725e-05,  5.19056025e-04,  7.89101162e-06,
       -3.12455627e-04, -1.02586119e-07])

In [ ]:
from scipy.special import jv
#from hhb.cat.hhb_cat import SnailParams, HBSettingsSnail, SnailHillMethod  # cell 0 inlines this module

w_a = 25.338776456203686
w_b = 2 * w_a
phi_a, phi_b = 0.11, 0.204
E_J = 37.12 * 2 * np.pi *1.5
kappa_b = 1 / 10.4
w_d = w_b
epsilon_p = 0.4
sinep = np.sin(epsilon_p)

g_bessel = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
epsilon_d = 8.12 * g_bessel

p = SnailParams(wa=w_a, wb=w_b, phia=phi_a, phib=phi_b, EJ=E_J, sinep=sinep,
                 epsd=epsilon_d, wd=w_d, kappab=kappa_b)

# nu=2 required: wb = 2*wa = wd means mode a's subharmonic must be representable
s = HBSettingsSnail(N_H=6, samples_per_harmonic=900, nu=2)
m = SnailHillMethod(p, s)

z0 = sol_test_cat_pos#np.random.rand(m.dim)#
sol = m.solve(z0)
print("residual norm:", np.linalg.norm(m.residual(sol)))

X_hb = m.x_tilde_to_X(m.Gamma @ sol)     # (14, N) over one period
mu_a = X_hb[0] + 1j * X_hb[1]
ta2 = X_hb[6] + 1j * X_hb[7]
a2_raw = mu_a**2 + ta2                    # <a^2> = mu_a^2 + tilde_sigma_a^2

g = 2 * E_J * sinep * phi_a**2 * phi_b
print("predicted |alpha|^2 = epsilon_d/g:", epsilon_d / g)
print("HB mean |<a^2>|:", np.mean(np.abs(a2_raw)))
print("HB mean |mu_a|^2:", np.mean(np.abs(mu_a)**2))

In [ ]:
def to_rotating_frame(X, tau, wa, wb):
    # X: (14, N) time-domain moments from x_tilde_to_X
    mu_a  = X[0] + 1j*X[1]
    mu_b  = X[2] + 1j*X[3]
    sa2   = X[4]
    sb2   = X[5]
    ta2   = X[6] + 1j*X[7]
    tb2   = X[8] + 1j*X[9]
    c     = X[10] + 1j*X[11]
    d     = X[12] + 1j*X[13]

    ea, eb = np.exp(1j*wa*tau), np.exp(1j*wb*tau)

    mu_a_r = mu_a * ea
    mu_b_r = mu_b * eb
    ta2_r  = ta2 * ea**2
    tb2_r  = tb2 * eb**2
    c_r    = c * ea * eb
    d_r    = d * ea * eb.conj()
    # sa2, sb2 unchanged

    return mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r

mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 2, figsize=(10, 20))

ax = axes[0, 0]
ax.scatter(mu_a_r.real, mu_a_r.imag)
ax.set_xlabel('t / T'); ax.set_title('mu_a(t)')

ax = axes[0, 1]
ax.scatter(mu_b_r.real, mu_b_r.imag)
ax.set_xlabel('t / T'); ax.set_title('mu_b(t)')

ax = axes[1, 0]
ax.plot(m.tau_j, sa2.imag)
ax.set_xlabel('t / T'); ax.set_title('sigma_sq_a(t)')

ax = axes[1, 1]
ax.plot(m.tau_j, sb2.imag)
ax.set_xlabel('t / T'); ax.set_title('sigma_sq_b(t)')

ax = axes[2, 0]
ax.scatter(ta2_r.real, ta2_r.imag)
ax.set_xlabel('t / T'); ax.set_title('sigma_t_sq_a(t)')

ax = axes[2, 1]
ax.scatter(tb2_r.real, tb2_r.imag)
ax.set_xlabel('t / T'); ax.set_title('sigma_t_sq_b(t)')

ax = axes[3, 0]
ax.scatter(c_r.real, c_r.imag)
ax.set_xlabel('t / T'); ax.set_title('c(t)')

ax = axes[3, 1]
ax.scatter(d_r.real, d_r.imag)
ax.set_xlabel('t / T'); ax.set_title('d(t)')

fig.tight_layout()
plt.plot()

In [ ]:
real_axis = np.linspace(-20, 20, 100)
imag_axis = np.linspace(-20, 20, 100)

fig, ax = plt.subplots()

# --- initial frame ---
def compute_W(idx_time):
    mu   = mu_a_r[idx_time]
    sig  = sa2[idx_time]
    sigt = ta2_r[idx_time]
    Z    = X + 1j * Y - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi / denom
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))

X, Y = np.meshgrid(real_axis, imag_axis)
W0   = compute_W(0)

mesh = ax.pcolormesh(real_axis, imag_axis, W0, cmap='RdBu_r')
plt.colorbar(mesh, ax=ax, label='W')
plt.show()

In [ ]:
import matplotlib.animation as animation
fig, ax = plt.subplots()
mesh = ax.pcolormesh(real_axis, imag_axis, W0, cmap='RdBu_r')
scat = ax.scatter(mu_a_r[0].real, mu_a_r[0].imag, color='black', s=20)
plt.colorbar(mesh, ax=ax, label='W')
ax.set_xlabel(r'Re$(\alpha)$')
ax.set_ylabel(r'Im$(\alpha)$')
title = ax.set_title('t = 0')

# --- update function ---
def update(idx_time):
    W = compute_W(idx_time)
    mesh.set_array(W.ravel())
    mesh.set_clim(W.min(), W.max())
    scat.set_offsets([mu_a_r[idx_time].real, mu_a_r[idx_time].imag])
    title.set_text(f't = {idx_time}')
    return mesh, scat, title

ani = animation.FuncAnimation(
    fig, update, frames=len(mu_a_r), interval=50, blit=True
)

plt.tight_layout()
plt.show()

#Optional: save as gif or mp4
ani.save(MEDIA_DIR / "wigner_test.gif", writer="pillow", fps=50)

In [ ]:
z_0_list[4]

In [ ]:
z_0_list = []
mu_a_r_list = []
mu_b_r_list = []
for k in range(100):
    print(k)
    z0 = -np.random.rand(m.dim)
    z_0_list.append(z0)
    try :
        sol = m.solve(z0)
        print("residual norm:", np.linalg.norm(m.residual(sol)))

        X_hb = m.x_tilde_to_X(m.Gamma @ sol)     # (14, N) over one period
        mu_a = X_hb[0] + 1j * X_hb[1]
        ta2 = X_hb[6] + 1j * X_hb[7]
        a2_raw = mu_a**2 + ta2                    # <a^2> = mu_a^2 + tilde_sigma_a^2

        g = 2 * E_J * sinep * phi_a**2 * phi_b
        print("predicted |alpha|^2 = epsilon_d/g:", epsilon_d / g)
        print("HB mean |<a^2>|:", np.mean(np.abs(a2_raw)))
        print("HB mean |mu_a|^2:", np.mean(np.abs(mu_a)**2))
        mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)
        mu_a_r_list.append(mu_a_r)
        mu_b_r_list.append(mu_b_r)
    except RuntimeError as e:
        print(f" failed: {e}")
        continue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

idx_limit_cycle = 15
mu_a_r = mu_a_r_list[idx_limit_cycle]
mu_b_r = mu_b_r_list[idx_limit_cycle]

axes[0].scatter(mu_a_r.real, mu_a_r.imag, s=.5)
axes[1].scatter(mu_b_r.real, mu_b_r.imag, s=.5)

axes[0].set_xlabel(r"$\text{Re}(\mu_a)$"); axes[0].set_ylabel(r"$\text{Im}(\mu_a)$")
axes[1].set_xlabel(r"$\text{Re}(\mu_b)$"); axes[1].set_ylabel(r"$\text{Im}(\mu_b)$")
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for (mu_a_r,mu_b_r) in zip(mu_a_r_list, mu_b_r_list):
    axes[0].scatter(mu_a_r.real, mu_a_r.imag, s=.5)
    axes[1].scatter(mu_b_r.real, mu_b_r.imag, s=.5)

axes[0].set_xlabel(r"$\text{Re}(\mu_a)$"); axes[0].set_ylabel(r"$\text{Im}(\mu_a)$")
axes[1].set_xlabel(r"$\text{Re}(\mu_b)$"); axes[1].set_ylabel(r"$\text{Im}(\mu_b)$")
fig.tight_layout()
plt.show()

In [ ]:
z0 = z_0_list[95]#sol_test_satellite_right
sol = m.solve(z0)
print("residual norm:", np.linalg.norm(m.residual(sol)))

X_hb = m.x_tilde_to_X(m.Gamma @ sol)     # (14, N) over one period
mu_a = X_hb[0] + 1j * X_hb[1]
ta2 = X_hb[6] + 1j * X_hb[7]
a2_raw = mu_a**2 + ta2                    # <a^2> = mu_a^2 + tilde_sigma_a^2

g = 2 * E_J * sinep * phi_a**2 * phi_b
print("predicted |alpha|^2 = epsilon_d/g:", epsilon_d / g)
print("HB mean |<a^2>|:", np.mean(np.abs(a2_raw)))
print("HB mean |mu_a|^2:", np.mean(np.abs(mu_a)**2))

mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)

In [ ]:
sol_test_satellite_left = sol

In [ ]:
real_axis = np.linspace(-10, 10, 100)
imag_axis = np.linspace(-10, 10, 100)

# --- initial frame ---
def compute_W(mu, sig, sigt):
    Z    = X + 1j * Y - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi / denom
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plt.subplots_adjust(wspace=0.5)
X, Y = np.meshgrid(real_axis, imag_axis)
idx_time = 0
mu_a   = mu_a_r[idx_time]
sig_a  = sa2[idx_time]
sigt_a = ta2_r[idx_time]
Wa   = compute_W(mu_a, sig_a, sigt_a)
mu_b   = mu_b_r[idx_time]
sig_b  = sb2[idx_time]
sigt_b = tb2_r[idx_time]
Wb   = compute_W(mu_b, sig_b, sigt_b)

mesh_a = axes[0].pcolormesh(real_axis, imag_axis, Wa, cmap='RdBu_r')
axes[0].set_title("Mode a", fontsize=25)
axes[0].tick_params(axis='both', labelsize=20)
cbar = fig.colorbar(mesh_a, ax=axes[0], fraction=0.05, pad=0.02)
cbar.set_label(r"$W_a$", fontsize=25)
cbar.ax.tick_params(labelsize=20)

mesh_b = axes[1].pcolormesh(real_axis, imag_axis, Wb, cmap='RdBu_r')
axes[1].set_title("Mode b")
axes[1].set_title("Mode b", fontsize=25)
axes[1].tick_params(axis='both', labelsize=20)
cbar = fig.colorbar(mesh_b, ax=axes[1], fraction=0.05, pad=0.02)
cbar.set_label(r"$W_b$", fontsize=25)
cbar.ax.tick_params(labelsize=20)

#plt.savefig(CAT_FIG_DIR / "hhb_wigner_satellite_left.pdf", bbox_inches="tight")
plt.show()

In [ ]:
real_axis = np.linspace(-10, 10, 100)
imag_axis = np.linspace(-10, 10, 100)

# --- initial frame ---
def compute_W(mu, sig, sigt):
    Z    = X + 1j * Y - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi / denom
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plt.subplots_adjust(wspace=0.5)
X, Y = np.meshgrid(real_axis, imag_axis)
idx_time = 0
mu_a   = mu_a_r[idx_time]
sig_a  = sa2[idx_time]
sigt_a = ta2_r[idx_time]
Wa   = compute_W(mu_a, sig_a, sigt_a)
mu_b   = mu_b_r[idx_time]
sig_b  = sb2[idx_time]
sigt_b = tb2_r[idx_time]
Wb   = compute_W(mu_b, sig_b, sigt_b)

mesh_a = axes[0].pcolormesh(real_axis, imag_axis, Wa, cmap='RdBu_r')
axes[0].set_title("Mode a", fontsize=25)
axes[0].tick_params(axis='both', labelsize=20)
cbar = fig.colorbar(mesh_a, ax=axes[0], fraction=0.05, pad=0.02)
cbar.set_label(r"$W_a$", fontsize=25)
cbar.ax.tick_params(labelsize=20)

mesh_b = axes[1].pcolormesh(real_axis, imag_axis, Wb, cmap='RdBu_r')
axes[1].set_title("Mode b")
axes[1].set_title("Mode b", fontsize=25)
axes[1].tick_params(axis='both', labelsize=20)
cbar = fig.colorbar(mesh_b, ax=axes[1], fraction=0.05, pad=0.02)
cbar.set_label(r"$W_b$", fontsize=25)
cbar.ax.tick_params(labelsize=20)

#plt.savefig(CAT_FIG_DIR / "hhb_wigner_cat_pos.pdf", bbox_inches="tight")
plt.show()

In [ ]:
beta_a_p = 2*E_J*np.sin(epsilon_p)*phi_a**2/w_a
beta_b_p = 2*E_J*np.sin(epsilon_p)*phi_b**2/w_b
kappa_b_p = kappa_b/w_b
epsilon_d_p = epsilon_d*phi_b/w_b
b_0 = 1/(12*(beta_a_p+beta_b_p))+np.sqrt((1/(12*(beta_a_p+beta_b_p)))**2+epsilon_d_p/3/beta_b_p)
gamma = 1+beta_b_p/beta_a_p
a_1 = b_0/(-2*gamma)
a_0 = np.sqrt(a_1/(-beta_a_p)-b_0**2)
theta_b_0 = -2*kappa_b_p*b_0/epsilon_d_p
theta_a_0 = theta_b_0/2

In [ ]:
from hhb.plot_setting import *
real_axis = np.linspace(-10, 10, 100)
imag_axis = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(real_axis, imag_axis)
idx_time = 0
def compute_W(mu, sig, sigt):
    Z    = X + 1j * Y - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi / denom
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))

sol_limit_cycle_list = [sol_test_cat_pos, sol_test_cat_neg, sol_test_satellite_left, sol_test_satellite_right]

fig, axes = plt.subplots(1, 2, figsize=(10, 10))
plt.subplots_adjust(wspace=0.4)
axes[0].scatter([0], [np.sqrt(epsilon_d/g)], marker="x", s=100, color=Perso_colormap(0))
axes[0].scatter([0], [-np.sqrt(epsilon_d/g)], marker="x", s=100, color=Perso_colormap(0))
axes[1].scatter([0], [0], marker="x", s=100, color=Perso_colormap(0))

axes[0].scatter([-a_0/phi_a/np.sqrt(2)*np.cos(theta_a_0)], [a_0/phi_a/np.sqrt(2)*np.sin(theta_a_0)], marker="x", s=100, color=Perso_colormap(125))
axes[0].scatter([a_0/phi_a/np.sqrt(2)*np.cos(theta_a_0)], [-a_0/phi_a/np.sqrt(2)*np.sin(theta_a_0)], marker="x", s=100, color=Perso_colormap(125))
axes[1].scatter([b_0/phi_b/np.sqrt(2)*np.cos(theta_b_0)], [b_0/phi_b/np.sqrt(2)*np.sin(theta_b_0)], marker="x", s=100, color=Perso_colormap(125))

for (i,sol) in enumerate(sol_limit_cycle_list):
    if i < 2: color_idx = 0
    else: color_idx = 125
    X_hb = m.x_tilde_to_X(m.Gamma @ sol)
    mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)
    axes[0].plot(mu_a_r.real, mu_a_r.imag, lw=4, color=Perso_colormap(color_idx))
    axes[1].plot(mu_b_r.real, mu_b_r.imag, lw=4, color=Perso_colormap(color_idx))


axes[0].set_xlabel(r"$\text{Re}(\mu_a)$", fontsize=25); axes[0].set_ylabel(r"$\text{Im}(\mu_a)$", fontsize=25)
axes[1].set_xlabel(r"$\text{Re}(\mu_b)$", fontsize=25); axes[1].set_ylabel(r"$\text{Im}(\mu_b)$", fontsize=25)
axes[0].set_ylim(-2,2)
axes[0].set_aspect('equal')
axes[0].tick_params(axis='both', labelsize=20)
axes[1].set_aspect('equal')
axes[1].tick_params(axis='both', labelsize=20)

#plt.savefig(CAT_FIG_DIR / "hhb_cat_limit_cycles.pdf", bbox_inches="tight")
plt.show()

In [ ]:
real_axis = np.linspace(-10, 10, 100)
imag_axis = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(real_axis, imag_axis)
idx_time = 0
def compute_W(mu, sig, sigt):
    Z    = X + 1j * Y - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi / denom
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))

sol_limit_cycle_list = [sol_test_cat_pos, sol_test_cat_neg, sol_test_satellite_left, sol_test_satellite_right]

fig, axes = plt.subplots(5, 2, figsize=(10, 40))
axes[0,0].scatter([0], [np.sqrt(epsilon_d/g)], marker="x")
axes[0,0].scatter([0], [-np.sqrt(epsilon_d/g)], marker="x")
axes[0,0].scatter([-a_0/phi_a/np.sqrt(2)*np.cos(theta_a_0)], [a_0/phi_a/np.sqrt(2)*np.sin(theta_a_0)], marker="x")
axes[0,0].scatter([a_0/phi_a/np.sqrt(2)*np.cos(theta_a_0)], [-a_0/phi_a/np.sqrt(2)*np.sin(theta_a_0)], marker="x")
axes[0,1].scatter([b_0/phi_b/np.sqrt(2)*np.cos(theta_b_0)], [b_0/phi_b/np.sqrt(2)*np.sin(theta_b_0)], marker="x")
for sol in sol_limit_cycle_list:
    X_hb = m.x_tilde_to_X(m.Gamma @ sol)
    mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)
    axes[0,0].plot(mu_a_r.real, mu_a_r.imag)
    axes[0,1].plot(mu_b_r.real, mu_b_r.imag)
    axes[1,0].plot(m.tau_j, sa2)
    axes[1,1].plot(m.tau_j, sb2)
    axes[2,0].plot(ta2_r.real, ta2_r.imag)
    axes[2,1].plot(tb2_r.real, tb2_r.imag)
    axes[3,0].plot(m.tau_j, sa2-np.abs(ta2_r)-1)
    axes[3,0].axhline(0, c='r')
    axes[3,1].plot(m.tau_j, sb2-np.abs(tb2_r)-1)
    axes[3,1].axhline(0, c='r')
    axes[4,0].plot(c_r.real, c_r.imag)
    axes[4,1].plot(d_r.real, d_r.imag)


axes[0,0].set_xlabel(r"$\text{Re}(\mu_a)$"); axes[0,0].set_ylabel(r"$\text{Im}(\mu_a)$")
axes[0,1].set_xlabel(r"$\text{Re}(\mu_b)$"); axes[0,1].set_ylabel(r"$\text{Im}(\mu_b)$")
axes[1,0].set_xlabel(r"$t$"); axes[1,0].set_ylabel(r"$\sigma_a^2$")
axes[1,1].set_xlabel(r"$t$"); axes[1,1].set_ylabel(r"$\sigma_b^2$")
axes[2,0].set_xlabel(r"$\text{Re}(\tilde{\sigma}_a^2)$"); axes[2,0].set_ylabel(r"$\text{Im}(\tilde{\sigma}_a^2)$")
axes[2,1].set_xlabel(r"$\text{Re}(\tilde{\sigma}_b^2)$"); axes[2,1].set_ylabel(r"$\text{Im}(\tilde{\sigma}_b^2)$")
axes[4,0].set_xlabel(r"$\text{Re}(c)$"); axes[4,0].set_ylabel(r"$\text{Im}(c)$")
axes[4,1].set_xlabel(r"$\text{Re}(d)$"); axes[4,1].set_ylabel(r"$\text{Im}(d)$")

axes[0,0].set_aspect('equal')
axes[0,1].set_aspect('equal')

fig.tight_layout()
plt.show()

for sol in sol_limit_cycle_list:
    X_hb = m.x_tilde_to_X(m.Gamma @ sol)     # (14, N) over one period
    mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)
    mu_a   = mu_a_r[idx_time]
    sig_a  = sa2[idx_time]
    sigt_a = ta2_r[idx_time]
    Wa   = compute_W(mu_a, sig_a, sigt_a)
    mu_b   = mu_b_r[idx_time]
    sig_b  = sb2[idx_time]
    sigt_b = tb2_r[idx_time]
    Wb   = compute_W(mu_b, sig_b, sigt_b)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    mesh_a = axes[0].pcolormesh(real_axis, imag_axis, Wa, cmap='RdBu_r')
    axes[0].set_title("Mode a")
    plt.colorbar(mesh_a, ax=axes[0], label='W_a')

    mesh_b = axes[1].pcolormesh(real_axis, imag_axis, Wb, cmap='RdBu_r')
    axes[1].set_title("Mode b")
    plt.colorbar(mesh_b, ax=axes[1], label='W_b')

    plt.show()

In [ ]:
import matplotlib.animation as animation
for (i,sol) in enumerate(sol_limit_cycle_list[2:]):
    X_hb = m.x_tilde_to_X(m.Gamma @ sol)
    mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)
    mu_a_r_gif=mu_a_r[::10]
    sa2_gif = sa2[::10]
    ta2_r_gif = ta2_r[::10]
    def compute_W(idx_time):
        mu   = mu_a_r_gif[idx_time]
        sig  = sa2_gif[idx_time]
        sigt = ta2_r[idx_time]
        Z    = X + 1j * Y - mu
        denom = sig**2 - np.abs(sigt)**2
        return (1/np.pi / denom
                * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))
    fig, ax = plt.subplots()
    mesh = ax.pcolormesh(real_axis, imag_axis, W0, cmap='RdBu_r')
    scat = ax.scatter(mu_a_r_gif[0].real, mu_a_r_gif[0].imag, color='black', s=20)
    plt.colorbar(mesh, ax=ax, label='W')
    ax.set_xlabel(r'Re$(\alpha)$')
    ax.set_ylabel(r'Im$(\alpha)$')
    title = ax.set_title('t = 0')

    # --- update function ---
    def update(idx_time):
        W = compute_W(idx_time)
        mesh.set_array(W.ravel())
        mesh.set_clim(W.min(), W.max())
        scat.set_offsets([mu_a_r_gif[idx_time].real, mu_a_r_gif[idx_time].imag])
        title.set_text(f't = {idx_time}')
        return mesh, scat, title

    ani = animation.FuncAnimation(
        fig, update, frames=len(mu_a_r_gif), interval=50, blit=True
    )

    plt.tight_layout()
    plt.show()

    #Optional: save as gif or mp4
    ani.save(MEDIA_DIR / ("wigner_test_" + str(i) + "_b.gif"), writer="pillow", fps=50)

In [ ]:
z_guess = sol_test_satellite_left
p = SnailParams(wa=w_a, wb=w_b, phia=phi_a, phib=phi_b, EJ=E_J, sinep=sinep,
                 epsd=epsilon_d, wd=w_d, kappab=kappa_b)

# nu=2 required: wb = 2*wa = wd means mode a's subharmonic must be representable
s = HBSettingsSnail(N_H=20, samples_per_harmonic=900, nu=2)
m = SnailHillMethod(p, s)

z_guess = np.pad(z_guess, (0,s.n*(2*s.N_H+1)-len(z_guess)), constant_values=0.0)
sol = m.solve(z_guess)

In [ ]:
import json
with open(CAT_DATA_DIR / "sol_satellite_left.json", "w") as f:
    json.dump(sol.tolist(), f)

In [ ]:
X_hb = m.x_tilde_to_X(m.Gamma @ sol)
mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 2, figsize=(10, 20))

ax = axes[0, 0]
ax.scatter(mu_a_r.real, mu_a_r.imag)
ax.set_xlabel('t / T'); ax.set_title('mu_a(t)')

ax = axes[0, 1]
ax.scatter(mu_b_r.real, mu_b_r.imag)
ax.set_xlabel('t / T'); ax.set_title('mu_b(t)')

ax = axes[1, 0]
ax.plot(m.tau_j, sa2 - np.abs(ta2_r) - 1)
ax.set_xlabel('t / T'); ax.set_title('sigma_sq_a(t)')

ax = axes[1, 1]
ax.plot(m.tau_j, sb2 - np.abs(tb2_r) - 1)
ax.set_xlabel('t / T'); ax.set_title('sigma_sq_b(t)')

ax = axes[2, 0]
ax.scatter(ta2_r.real, ta2_r.imag)
ax.set_xlabel('t / T'); ax.set_title('sigma_t_sq_a(t)')

ax = axes[2, 1]
ax.scatter(tb2_r.real, tb2_r.imag)
ax.set_xlabel('t / T'); ax.set_title('sigma_t_sq_b(t)')

ax = axes[3, 0]
ax.scatter(c_r.real, c_r.imag)
ax.set_xlabel('t / T'); ax.set_title('c(t)')

ax = axes[3, 1]
ax.scatter(d_r.real, d_r.imag)
ax.set_xlabel('t / T'); ax.set_title('d(t)')

fig.tight_layout()
plt.plot()

In [ ]:
real_axis = np.linspace(-10, 10, 100)
imag_axis = np.linspace(-10, 10, 100)

# --- initial frame ---
def compute_W(mu, sig, sigt):
    Z    = X + 1j * Y - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi / denom
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
X, Y = np.meshgrid(real_axis, imag_axis)
idx_time = 3000
mu_a   = mu_a_r[idx_time]
sig_a  = sa2[idx_time]
sigt_a = ta2_r[idx_time]
Wa   = compute_W(mu_a, sig_a, sigt_a)
mu_b   = mu_b_r[idx_time]
sig_b  = sb2[idx_time]
sigt_b = tb2_r[idx_time]
Wb  = compute_W(mu_b, sig_b, sigt_b)

mesh_a = axes[0].pcolormesh(real_axis, imag_axis, Wa, cmap='RdBu_r')
axes[0].set_title("Mode a")
plt.colorbar(mesh_a, ax=axes[0], label='W_a')

mesh_b = axes[1].pcolormesh(real_axis, imag_axis, Wb, cmap='RdBu_r')
axes[1].set_title("Mode b")
plt.colorbar(mesh_b, ax=axes[1], label='W_b')

plt.show()

In [ ]:
import matplotlib.animation as animation
mu_a_r_gif=mu_b_r[::10]
sa2_gif = sb2[::10]
ta2_r_gif = tb2_r[::10]
def compute_W(idx_time):
    mu   = mu_a_r_gif[idx_time]
    sig  = sa2_gif[idx_time]
    sigt = ta2_r_gif[idx_time]
    Z    = X + 1j * Y - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi / denom
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))
fig, ax = plt.subplots()
W0 = compute_W(0)
mesh = ax.pcolormesh(real_axis, imag_axis, W0, cmap='RdBu_r')
scat = ax.scatter(mu_a_r_gif[0].real, mu_a_r_gif[0].imag, color='black', s=20)
plt.colorbar(mesh, ax=ax, label='W')
ax.set_xlabel(r'Re$(\alpha)$')
ax.set_ylabel(r'Im$(\alpha)$')
title = ax.set_title('t = 0')

# --- update function ---
def update(idx_time):
    W = compute_W(idx_time)
    mesh.set_array(W.ravel())
    mesh.set_clim(W.min(), W.max())
    scat.set_offsets([mu_a_r_gif[idx_time].real, mu_a_r_gif[idx_time].imag])
    title.set_text(f't = {idx_time}')
    return mesh, scat, title

ani = animation.FuncAnimation(
    fig, update, frames=len(mu_a_r_gif), interval=50, blit=True
)

plt.tight_layout()
plt.show()

#Optional: save as gif or mp4
ani.save(MEDIA_DIR / "wigner_test_satellite_left_b.gif", writer="pillow", fps=50)

In [ ]:
import matplotlib.animation as animation

X_hb = m.x_tilde_to_X(m.Gamma @ sol_test_satellite_right)
mu_a_r, mu_b_r, sa2, sb2, ta2_r, tb2_r, c_r, d_r = to_rotating_frame(X_hb, m.tau_j, w_a, w_b)

real_axis = np.linspace(-10, 10, 100)
imag_axis = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(real_axis, imag_axis)

gap_frame = 20
mu_a_r_gif = mu_a_r[::gap_frame]
sa2_gif    = sa2[::gap_frame]
ta2_r_gif  = ta2_r[::gap_frame]

mu_b_r_gif = mu_b_r[::gap_frame]
sb2_gif    = sb2[::gap_frame]
tb2_r_gif  = tb2_r[::gap_frame]

def compute_H(mu, sig, sigt):
    Z    = X + 1j * Y - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1/np.pi / denom
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))

# --- Initial frame ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plt.subplots_adjust(wspace=0.5)

Ha0 = compute_H(mu_a_r_gif[0], sa2_gif[0], ta2_r_gif[0])
Hb0 = compute_H(mu_b_r_gif[0], sb2_gif[0], tb2_r_gif[0])

mesh_a = axes[0].pcolormesh(real_axis, imag_axis, Ha0, cmap='RdBu_r')
trail_a = axes[0].scatter(mu_a_r_gif.real, mu_a_r_gif.imag, color='gray', s=5, alpha=0.4)  # static full trail
point_a = axes[0].scatter(mu_a_r_gif[0].real, mu_a_r_gif[0].imag, color='black', s=10, zorder=5)  # moving marker
axes[0].set_title("Mode a", fontsize=25)
axes[0].tick_params(axis='both', labelsize=20)
cbar_a = fig.colorbar(mesh_a, ax=axes[0], fraction=0.05, pad=0.02)
cbar_a.set_label(r"$H_a$", fontsize=25)
cbar_a.ax.tick_params(labelsize=20)

mesh_b = axes[1].pcolormesh(real_axis, imag_axis, Hb0, cmap='RdBu_r')
trail_b = axes[1].scatter(mu_b_r_gif.real, mu_b_r_gif.imag, color='gray', s=5, alpha=0.4)  # static full trail
point_b = axes[1].scatter(mu_b_r_gif[0].real, mu_b_r_gif[0].imag, color='black', s=10, zorder=5)  # moving marker
axes[1].set_title("Mode b", fontsize=25)
axes[1].tick_params(axis='both', labelsize=20)
cbar_b = fig.colorbar(mesh_b, ax=axes[1], fraction=0.05, pad=0.02)
cbar_b.set_label(r"$H_b$", fontsize=25)
cbar_b.ax.tick_params(labelsize=20)

title = fig.suptitle('t = 0', fontsize=22, y=1.05)

# --- Update function ---
def update(idx_time):
    Ha = compute_H(mu_a_r_gif[idx_time], sa2_gif[idx_time], ta2_r_gif[idx_time])
    Hb = compute_H(mu_b_r_gif[idx_time], sb2_gif[idx_time], tb2_r_gif[idx_time])

    mesh_a.set_array(Ha.ravel())
    mesh_a.set_clim(Ha.min(), Ha.max())
    point_a.set_offsets([mu_a_r_gif[idx_time].real, mu_a_r_gif[idx_time].imag])

    mesh_b.set_array(Hb.ravel())
    mesh_b.set_clim(Hb.min(), Hb.max())
    point_b.set_offsets([mu_b_r_gif[idx_time].real, mu_b_r_gif[idx_time].imag])

    title.set_text(f't = {idx_time}')
    return mesh_a, point_a, mesh_b, point_b, title

ani = animation.FuncAnimation(
    fig, update, frames=len(mu_a_r_gif), interval=50, blit=False
)

plt.show()

# Optional: save as gif or mp4
ani.save(MEDIA_DIR / "wigner_two_modes_satellite_right.gif", writer="pillow", fps=50)